In [6]:
import itertools

import pandas as pd
import torch
from matplotlib import pyplot as plt
from torch.utils.data.dataloader import DataLoader

from homework_3.classes import ClipModel, ApiResponsesDataset

ImportError: cannot import name 'ApiResponsesDataset' from 'homework_3.classes' (/home/Baldi/Projects/PycharmProjects/uni-MultiModal/homework_3/classes.py)

In [ ]:
# TODO Add validation
def train_model(model, optimizer, criterion, train_data, num_epochs):
    losses = []
    for epoch in range(num_epochs):
        print(f"Epoch {epoch + 1}/{num_epochs}")
        epoch_loss = 0.0
        for data, labels in train_data:
            if torch.cuda.is_available():
                data = data.cuda()
                labels = labels.cuda()
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(train_data))
        print(f"Train loss: {losses[-1]}")

    plt.plot(losses)
    plt.title("Loss curve")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.show()

    return model



In [ ]:
WEIGHTS_DIR = "../weights"
RANDOM_SEED = 161
BATCH_SIZE = 2
LEARNING_RATE = 1e-4
NUM_EPOCHS = 25
NAME = f"{NUM_EPOCHS}_{BATCH_SIZE}"

torch.autograd.set_detect_anomaly(True, check_nan=False)
torch.random.manual_seed(RANDOM_SEED)

clip_model = ClipModel()

df = pd.read_parquet("../datasets/cifar100_train.parquet")
series = df.img[:100]

dataset = ApiResponsesDataset(clip_model.preprocess, img_series=series)
train_dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

adam_optimizer = torch.optim.Adam(
    itertools.chain(clip_model.head_concept1.parameters(), clip_model.head_concept2.parameters()), lr=LEARNING_RATE)

crossEntropyLoss = torch.nn.CrossEntropyLoss()

In [ ]:
trained_model = train_model(model=clip_model,
                            optimizer=adam_optimizer,
                            criterion=crossEntropyLoss,
                            train_data=train_dataloader,
                            num_epochs=NUM_EPOCHS)

torch.save(trained_model.head_concept1.state_dict(), f"{WEIGHTS_DIR}/concept1_{NAME}.pth")
torch.save(trained_model.head_concept2.state_dict(), f"{WEIGHTS_DIR}/concept2_{NAME}.pth")

